<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Calibration images, 8×6 internal-corner geometry, 0.03 m square size |
| **Final outputs** | Intrinsic matrix K, one (R,t) pose per retained view, reprojection metrics, six diagnostic figures |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Open the lab with the project environment selected in VS Code.
2. Treat this notebook as the coding plan for `main.ipynb`; do not compute results here.
3. Proceed only in the order below because later calibration quantities depend on earlier validated geometry.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create the Setup

**Purpose**

Fix the calibration geometry and numerical environment.

**Inputs**

Python environment; 8 by 6 internal corners.

**Procedure**

1. Import `Path`, OpenCV, NumPy, and Matplotlib.
2. Set NumPy display precision so matrices and residuals are readable.
3. Define square size, 8×6 internal corners, minimum valid views, and corner-refinement criteria.

**Outputs**

Imports; 0.03 m spacing; minimum 3 views.

**Validation**

Check the board uses internal corners, not numbers of squares.

**On failure**

Stop on missing dependencies or an incorrect board specification.

## 2. Create Repository-Relative Paths

**Purpose**

Keep the experiment portable within the lab.

**Inputs**

Lab notebooks directory.

**Procedure**

1. Create the calibration-image path and `outputs/figures/` path.
2. Create the output directory if needed.
3. Define supported image extensions; use no machine-specific absolute path.

**Outputs**

DATA_DIR and OUTPUT_DIR paths.

**Validation**

Input directory exists; output directory is writable.

**On failure**

Resolve the working directory before loading data.

## 3. Discover and Validate Calibration Images

**Purpose**

Build a deterministic image inventory.

**Inputs**

Supported files under data/calibration_images.

**Procedure**

1. Enumerate supported image files and sort them deterministically.
2. If no image is found, stop with `FileNotFoundError`.
3. Create containers for readable images and retained calibration views.

**Outputs**

Sorted paths and readable_calibration_views.

**Validation**

At least one readable image; retain names alongside arrays.

**On failure**

Log and skip unreadable files; stop if none remain.

## 4. Detect and Refine Chessboard Corners

**Purpose**

Obtain consistent subpixel correspondences.

**Inputs**

Readable BGR images; REFINE_CRITERIA.

**Procedure**

1. For each image: read it and convert it to grayscale.
2. Detect the complete 8×6 chessboard pattern.
3. If found, refine corners with `cornerSubPix`; otherwise reject that view.
4. Store image name, image array, and refined 2-D points.
5. Stop if fewer than the required number of valid views remain.

**Outputs**

views with im_pts shaped (48, 2), in pixels.

**Validation**

Complete 8 by 6 pattern; at least 3 retained views.

**On failure**

Skip incomplete detections explicitly; stop below MIN_VALID_VIEWS.

## 5. Build Metric Planar Coordinates

**Purpose**

Attach metric board coordinates to detected corners.

**Inputs**

Board dimensions and SQUARE_SIZE_M.

**Procedure**

1. Create 48 planar points in the same ordering as detected corners.
2. Set each point to `(x·square_size, y·square_size)` in metres.
3. Keep the board on `Z=0`; add the zero z-coordinate only when 3-D points are needed.

**Outputs**

plane_pts shaped (48, 2), in metres.

**Validation**

x varies fastest, matching corner ordering; board lies on Z=0.

**On failure**

Resolve ordering or units before estimating homographies.

## 6. Define Homogeneous-Coordinate Conversion

**Purpose**

Enable homogeneous matrix multiplication.

**Inputs**

Plane/image points shaped (N, 2).

**Procedure**

1. Create a helper that appends a homogeneous coordinate of 1 to every 2-D point.
2. Use it for both plane and image correspondences.
3. Validate the returned shape before matrix computations.

**Outputs**

Homogeneous arrays shaped (N, 3).

**Validation**

Last column equals 1; first two columns preserve coordinates.

**On failure**

Do not pass arrays with missing or extra coordinate columns.

## 7. Normalize Plane and Image Points

**Purpose**

Balance coordinate magnitudes before solving DLT.

**Inputs**

48 plane and image points per view.

**Procedure**

1. Compute each point-set centroid and shift to zero mean.
2. Compute mean Euclidean distance from the centroid.
3. Set scale to `sqrt(2)/mean_distance`; reject coincident-point configurations.
4. Build each 3×3 normalization transform and apply it.
5. Store `T_plane`, `T_image`, and normalized point sets per view.

**Outputs**

T_plane, T_image and normalized points.

**Validation**

Centroid approximately zero; mean radius approximately sqrt(2). Current coincidence guard: mean distance below 1e-12.

**On failure**

Stop on coincident points; a large normalization scale alone is not degeneracy.

## 8. Build the Normalized DLT System

**Purpose**

Estimate a planar mapping up to scale.

**Inputs**

Normalized correspondences.

**Procedure**

1. For every correspondence add the two standard DLT rows to matrix `Q`.
2. Solve `Qh=0` by SVD.
3. Select the right-singular vector for the smallest singular value.
4. Reshape it into a normalized 3×3 homography.

**Outputs**

Q shaped (96, 9); singular values; H_normalized shaped (3, 3).

**Validation**

Finite Q and homography; use the last right-singular vector. An exact solution is expected to have a near-zero last singular value.

**On failure**

Inspect degenerate geometry if the solution is not isolated; do not reject an exact fit because sigma_max/sigma_min is large.

## 9. Denormalize Every Homography

**Purpose**

Express the homography in metres-to-pixels coordinates.

**Inputs**

H_normalized and both normalization matrices.

**Procedure**

1. Compute `H = inv(T_image) @ H_normalized @ T_plane`.
2. Normalize by `H[2,2]`; reject numerically degenerate scale.
3. Store one finite 3×3 homography per retained view.
4. Recheck that each view still contains 48 plane/image correspondences.

**Outputs**

One H shaped (3, 3) per view.

**Validation**

Current helper rejects abs(H[2,2]) below 1e-12 before scaling; verify 48 paired points.

**On failure**

Stop on the helper's scale error. This is a chosen normalization limitation, not proof the plane is parallel to the image.

## 10. Build Zhang Multi-View Constraints

**Purpose**

Combine orthogonality constraints from multiple views.

**Inputs**

Retained homographies.

**Procedure**

1. Extract `h1` and `h2` from every homography.
2. Implement the Zhang `v_ij` vector.
3. Stack `v12` and `v11-v22` for all views into matrix `V`.
4. Solve `Vb=0` by SVD and retain the last right-singular vector `b`.

**Outputs**

V shaped (2*number_of_views, 6); unit-norm b shaped (6,).

**Validation**

Inspect the singular spectrum and residual in the same scaling as V. Current implementation stacks unnormalized rows.

**On failure**

Do not infer failure from an absolute residual alone or require noisy data to reach machine precision.

## 11. Recover the Intrinsic Matrix

**Purpose**

Recover the camera's intrinsic parameters.

**Inputs**

b in order (B11,B12,B22,B13,B23,B33).

**Procedure**

1. Unpack `b` into the six symmetric-conic coefficients in the order used by `build_zhang_constraint_vector`.
2. Compute `denominator = b11*b22-b12**2`; the current code raises when its absolute value is below `1e-12`.
3. Compute `v0` and `lambda_`, then the two focal-length radicands.
4. Describe the existing sign-retry branch accurately: it negates `b` and recomputes the expressions, but the radicands are mathematically invariant to a global sign flip.
5. Assemble `K` with the closed-form formulas. Review finiteness and positive focal lengths before interpreting the result; the limitations section distinguishes this review from the current automatic checks.

**Outputs**

alpha, beta, gamma, u0, v0 and K shaped (3, 3).

**Validation**

Review finite radicands and positive focal lengths. Existing final check verifies only K shape and finiteness.

**On failure**

Stop analysis if K is nonphysical; sign reversal cannot repair negative radicands, whose ratios are sign-invariant. See implementation limits below.

## 12. Recover One Camera Pose per View

**Purpose**

Recover board-to-camera orientation and translation.

**Inputs**

K and each H.

**Procedure**

1. Compute `K^-1 h1`, `K^-1 h2`, and `K^-1 h3`.
2. Use the norm of `K^-1 h1` for the common pose scale.
3. Form `r1`, `r2`, `r3=r1×r2`, and `t`.
4. Project the approximate rotation to the nearest orthonormal matrix with SVD.
5. If `det(R)<0`, correct the sign so `R` is a proper rotation.
6. Store finite `R` and `t`.

**Outputs**

R shaped (3, 3); t shaped (3,), in metres.

**Validation**

Current scale is 1/norm(K^-1 h1); SVD projection enforces a proper rotation. Check orthogonality and determinant near +1.

**On failure**

Do not describe t_z>0 as enforced: current code has no cheirality correction.

## 13. Reproject Calibration Points

**Purpose**

Predict image coordinates from the recovered camera.

**Inputs**

K, R, t; board coordinates on Z=0.

**Procedure**

1. Convert planar points to `(X,Y,0)`.
2. Transform them into the camera frame with `R` and `t`.
3. Project with `K` and divide by homogeneous depth.
4. Reject near-zero depth and store the projected 2-D points.

**Outputs**

world_xyz shaped (48, 3); projected shaped (48, 2).

**Validation**

Current guard checks absolute homogeneous depth against 1e-12. Review positive camera-frame depth separately.

**On failure**

Stop on near-zero depth; flag negative depths as physically invalid even if division is finite.

## 14. Compute Calibration Errors

**Purpose**

Measure calibration fit in pixel units.

**Inputs**

Observed and projected points with identical ordering.

**Procedure**

1. Compute residual vectors `projected-observed`.
2. Convert residuals to point-wise Euclidean errors.
3. Compute per-view mean error and RMSE.
4. Concatenate all errors and compute overall mean error and RMSE.

**Outputs**

Per-point errors; per-view and global mean/RMSE.

**Validation**

Euclidean error per point; RMSE=sqrt(mean(error**2)); global metrics pool all points.

**On failure**

Investigate nonfinite values or systematic residuals; no universal pixel acceptance threshold is implemented.

## 15. Generate the Required Diagnostics

**Purpose**

Expose detection, geometry and residual quality visually.

**Inputs**

Stored views, K, poses and error arrays.

**Procedure**

1. Save the homography-estimation pipeline figure.
2. Save detected corners, camera poses, and reprojection overlays.
3. Save mean reprojection error by view and the global error distribution.
4. Build every plot from the same validated arrays used for numerical metrics.

**Outputs**

Six PNG diagnostics listed in the output contract.

**Validation**

Overlays and plots must use the same retained views and errors; camera centre is -R.T@t.

**On failure**

Regenerate missing plots; do not treat visually plausible overlays as physical validation.

## 16. Run Final Validation

**Purpose**

Check the implemented numerical and artifact conditions.

**Inputs**

Computed calibration and REQUIRED_OUTPUTS.

**Procedure**

1. Follow the final validation cell in `main.ipynb`: check finite `K` with shape `(3,3)`.
2. Check finite homographies, rotations with `R.T @ R` close to identity and determinant close to `+1`, and finite error arrays.
3. Check that all six declared figures exist.
4. Interpret the success message as the result of these implemented checks. Positive depths, finite translations and reprojection-quality thresholds are additional review requirements, not checks currently enforced by that cell.

**Outputs**

Final validation outcome.

**Validation**

K finite 3 by 3; H finite; R.T@R near identity and det(R) near 1; finite errors; six files present.

**On failure**

Any existing failed check stops execution; separately report unimplemented physical checks.

## Complete Formal Pseudocode

**Input:** Calibration images; an 8×6 internal-corner board with 0.03 m spacing.

**Output:** K ∈ R^(3×3); for each retained view R ∈ R^(3×3), t ∈ R^3; projected points ∈ R^(48×2); point errors; global mean/RMSE; six figures.

**Data contract:** Board points use metres and image points use pixels. Homographies map board coordinates to image coordinates. R and t map board coordinates into the camera frame.

The following numbered blocks form **one complete execution algorithm**. They include every lab-specific helper body, every numerical expression, every loop/branch and the existing validation/diagnostic steps. Inputs, literal parameters and output paths are assigned explicitly in the setup blocks; intermediate variables are produced before use in execution order. Each procedure's `RETURN` identifies its result. Top-level results are named by the output contract above.

**Scope:** this is a formal transcription of the current executable workflow, including its explicitly documented limitations. Additional physical-validity requirements in the surrounding review text are not silently presented as implemented guards. Error branches below specify the actual termination/rejection behaviour; the implementation-limit section identifies missing protections.

### Pseudocode notation and execution rules

- `x ← expression` assigns a value; `=` compares values. `A · B` is matrix multiplication; `*`, `/` and `^` are elementwise arithmetic for arrays and scalar arithmetic for numbers. Explicit dtype conversions are retained because clipping and overflow are part of these labs.
- Arrays are **zero-indexed**. `a:b:s` is an end-exclusive slice with step `s`; omitted limits use the array boundary. `RANGE(n)` produces `0,…,n−1`; `RANGE(a,b,s)` excludes `b`. Tuple assignment unpacks corresponding values. `TRANSPOSE(A)` interchanges axes as the array transpose operation does.
- `LIST[expression FOR x IN sequence WHERE condition]` constructs a list in iteration order. `STREAM[...]` is lazy; `SET{...}` deduplicates; `MAP{key ↦ value FOR ...}` creates a mapping. `EXPAND` inserts a sequence or mapping's members. `FUNCTION(parameters) ↦ expression` defines a callable expression.
- `IF`, `FOR`, `WHILE`, `RETURN`, `BREAK` and `CONTINUE` have their usual structured meanings and explicit closing statements. `REQUIRE condition OTHERWISE ...` stops on failure. `RAISE Error(message)` propagates that error; it is not a successful return. A logged frame/view rejection followed by `CONTINUE` is deliberately different from stopping the experiment.
- `AND`/`OR` short-circuit scalar conditions. `AND_EACH`/`OR_EACH` are elementwise operations for boolean masks and bitwise operations for integer flags. `COMPLEMENT` complements boolean masks or integer bits according to dtype. `NULL`, `TRUE`, `FALSE`, `ARRAY.nan` and `ARRAY.inf` retain their distinct meanings; `IS` tests identity.
- `PROCEDURE` defines the complete body shown below; indentation determines scope. Default arguments refer to the environment at definition time. `CLASS` defines stateful objects, `self` is the current instance, `__init__` initializes it, and `STATIC` methods need no instance. `NO_GRADIENT` disables automatic differentiation throughout the procedure. Each block continues the shared state of the preceding block; run them **once, top to bottom**, starting with a fresh environment.
- `CAST`, `RESHAPE`, `FLATTEN`, `COPY`, `MEAN`, `SUM`, `MIN`, `MAX`, `STD`, `APPEND`, `EXTEND`, `ITEMS`, `KEYS` and `VALUES` are standard array/container operations. `CAST` defaults to a new array; `FLATTEN` follows row-major order; axis arguments specify reductions. `TO_DEVICE`, `TO_CPU`, `TO_ARRAY`, `SCALAR_VALUE`, `ELEMENT_COUNT` and `TO_LIST` perform the named tensor/container conversions. `SET_MODE(model,TRAINING/EVALUATION)` switches neural-network behaviour.
- `LENGTH`, `SORT`, `ZIP`, `ENUMERATE`, `ABS`, `ALL`, `ANY`, `INTEGER`, `REAL`, `STRING`, `BOOLEAN`, `TUPLE`, `LIST`, `SET`, `MAP`, `GET_ATTRIBUTE`, `IS_INSTANCE` and `ROUND` are standard built-ins with the indicated roles. `super().__init__()` initializes the declared base class. `FORMAT` uses numbered placeholders and the explicitly retained numeric format specifications; `EMIT` writes a diagnostic. These are presentation operations, not hidden numerical algorithms.

### Standard-library primitive boundary

Calls under the following namespaces are explicitly delegated to standard libraries. Their named parameters and constants are shown at each call; all **lab-specific** procedures are defined in full below. Constructors, file objects, axes, video captures, optimizers and tensors retain the methods of their declared library. This fixes interpolation, boundary, dtype and random-state semantics without pretending to reimplement SVD, FFT, ORB, automatic differentiation or image codecs.

| Pseudocode namespace | Library / primitive family |
| --- | --- |
| `ARRAY` | NumPy arrays, broadcasting, linear algebra/SVD, FFT, statistics, RNG and window views |
| `CV` | OpenCV image conversion, chessboard detection/subpixel refinement, ORB/Hamming matching, RANSAC, morphology and watershed |
| `NDIMAGE`, `INTERPOLATE_COORDINATES` | SciPy ndimage filters, morphology and coordinate interpolation |
| `TENSOR`, `LAYERS` | PyTorch tensors, devices, differentiation, optimizers, layers and cross-entropy |
| `DATASET_INTERFACE`, `BATCH_LOADER` | PyTorch Dataset protocol and DataLoader batching, with the explicit local collation method |
| `READ_MNIST_IDX` | python-mnist IDX reader; `load_training` and `load_testing` return images and labels |
| `IMAGE` | Pillow image loading, mode conversion, resizing and encoding |
| `PATH` | Pathlib path joining, traversal, existence checks and output-directory creation; `/` joins path components for path operands |
| `PLOT` | Matplotlib figures, axes, drawing and saving; named arrays are the plotted data |
| `MATH`, `PROGRESS` | Standard scalar math and tqdm progress iteration |

`DATASET_INTERFACE` and `BATCH_LOADER` are standard abstractions; `MNISTDataset` and `MNISTClassifier`, where used, are fully defined below. Visualization calls preserve the figures' data, labels and output filenames. They can be translated to another renderer without changing the numerical procedure.

### Algorithm 1 — Setup — Environment and Configuration

```text
CALL ARRAY.set_printoptions(precision = 6, suppress = TRUE)
SQUARE_SIZE_M ← 0.03
INTERNAL_CORNERS_X ← 8
INTERNAL_CORNERS_Y ← 6
MIN_VALID_VIEWS ← 3
OUTPUT_DIR ← PATH("../outputs/figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
```

### Algorithm 2 — 1. Load the Sorted JPEG Calibration Images

```text
DATA_DIR ← PATH("../data/calibration_images")
IMAGE_EXTENSIONS ← {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
IF (NOT DATA_DIR.is_dir()) THEN
    RAISE FileNotFoundError(FORMAT("Calibration image directory not found: {0}" , DATA_DIR))
END IF
calibration_image_paths ← SORT(STREAM[path FOR path IN DATA_DIR.iterdir() WHERE (path.is_file() AND (path.suffix.lower() IN IMAGE_EXTENSIONS))])
IF (NOT calibration_image_paths) THEN
    RAISE FileNotFoundError(FORMAT("No supported calibration images found in: {0}" , DATA_DIR))
END IF
readable_calibration_views ← []
views ← []
FOR image_path IN calibration_image_paths DO
    image ← CV.imread(STRING(image_path))
    IF (image IS NULL) THEN
        CALL EMIT(FORMAT("[SKIPPED] {0}: image could not be read" , image_path.name))
        CONTINUE
    END IF
    CALL APPEND(readable_calibration_views, {"path" ↦ image_path, "name" ↦ image_path.name,
        "image" ↦ image})
END FOR
IF (NOT readable_calibration_views) THEN
    RAISE RuntimeError("No readable calibration images are available.")
END IF
CALL EMIT(FORMAT("Calibration files found : {0}" , LENGTH(calibration_image_paths)))
CALL EMIT(FORMAT("Readable views          : {0}" , LENGTH(readable_calibration_views)))
```

### Algorithm 3 — 2. Detect and Refine Chessboard Corners

```text
REFINE_CRITERIA ← ((CV.TERM_CRITERIA_EPS + CV.TERM_CRITERIA_MAX_ITER), 30, 0.001)
PROCEDURE detect_and_refine_chessboard_corners(image, corners_x = INTERNAL_CORNERS_X,
    corners_y = INTERNAL_CORNERS_Y)
    gray ← CV.cvtColor(image, CV.COLOR_BGR2GRAY)
    (found, corners) ← CV.findChessboardCorners(gray, (corners_x, corners_y), NULL)
    IF ((NOT found) OR (corners IS NULL)) THEN
        RETURN NULL
    END IF
    refined ← CV.cornerSubPix(gray, corners, (11, 11), ((−1), (−1)), REFINE_CRITERIA)
    RETURN RESHAPE(refined, (−1), 2)
END PROCEDURE
views ← []
FOR readable_calibration_view IN readable_calibration_views DO
    detected_corner_points ← detect_and_refine_chessboard_corners(readable_calibration_view["image"])
    IF (detected_corner_points IS NULL) THEN
        CALL EMIT(FORMAT("[SKIPPED] {0}: chessboard not detected" , readable_calibration_view["name"]))
        CONTINUE
    END IF
    view ← COPY(readable_calibration_view)
    view["im_pts"] ← detected_corner_points
    CALL APPEND(views, view)
    CALL EMIT(FORMAT("[OK] {0}: {1} refined corners" , view["name"], detected_corner_points.shape[0]))
END FOR
IF (LENGTH(views) < MIN_VALID_VIEWS) THEN
    RAISE RuntimeError(FORMAT("At least {0} valid calibration views are required; only {1} were detected." ,
        MIN_VALID_VIEWS, LENGTH(views)))
END IF
CALL EMIT(FORMAT("\nValid calibration views: {0}" , LENGTH(views)))
```

### Algorithm 4 — 3. Build the Planar World Coordinates

```text
PROCEDURE build_chessboard_plane_points(corners_x = INTERNAL_CORNERS_X, corners_y = INTERNAL_CORNERS_Y,
    square_size = SQUARE_SIZE_M)
    points ← ARRAY.zeros(((corners_x * corners_y), 2), dtype = ARRAY.float64)
    index ← 0
    FOR y_index IN RANGE(corners_y) DO
        FOR x_index IN RANGE(corners_x) DO
            points[(index, 0)] ← (x_index * square_size)
            points[(index, 1)] ← (y_index * square_size)
            index ← (index + 1)
        END FOR
    END FOR
    RETURN points
END PROCEDURE
chessboard_plane_points ← build_chessboard_plane_points()
FOR view IN views DO
    view["plane_pts"] ← COPY(chessboard_plane_points)
END FOR
CALL EMIT(FORMAT("Planar points per view: {0}" , chessboard_plane_points.shape[0]))
CALL EMIT(FORMAT("Square size: {0:.3f} m" , SQUARE_SIZE_M))
```

### Algorithm 5 — 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

```text
PROCEDURE convert_to_homogeneous_coordinates(points)
    points ← ARRAY.asarray(points, dtype = REAL)
    homogeneous ← ARRAY.ones((points.shape[0], 3), dtype = REAL)
    homogeneous[(:, :2)] ← points
    RETURN homogeneous
END PROCEDURE
PROCEDURE compute_point_normalization_transform(points)
    points ← ARRAY.asarray(points, dtype = REAL)
    centroid ← MEAN(ARRAY, points, axis = 0)
    shifted ← (points − centroid)
    distances ← ARRAY.linalg.norm(shifted, axis = 1)
    mean_distance ← MEAN(ARRAY, distances)
    IF (mean_distance < 1e-12) THEN
        RAISE ValueError("Point normalization is undefined for coincident points.")
    END IF
    scale ← (ARRAY.sqrt(2.0) / mean_distance)
    RETURN ARRAY.array([[scale, 0.0, ((−scale) * centroid[0])], [0.0, scale, ((−scale) * centroid[1])], [0.0,
        0.0, 1.0]], dtype = REAL)
END PROCEDURE
FOR view IN views DO
    T_image ← compute_point_normalization_transform(view["im_pts"])
    T_plane ← compute_point_normalization_transform(view["plane_pts"])
    view["T_image"] ← T_image
    view["T_plane"] ← T_plane
    view["normalized_image"] ← TRANSPOSE((T_image · TRANSPOSE(convert_to_homogeneous_coordinates(view["im_pts"]))))
    view["normalized_plane"] ← TRANSPOSE((T_plane · TRANSPOSE(convert_to_homogeneous_coordinates(view["plane_pts"]))))
END FOR
CALL EMIT("Image and planar point normalization completed.")
```

### Algorithm 6 — 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

```text
PROCEDURE build_homography_dlt_matrix(normalized_plane, normalized_image)
    n_points ← normalized_image.shape[0]
    Q ← ARRAY.zeros(((2 * n_points), 9), dtype = REAL)
    FOR i IN RANGE(n_points) DO
        (X, Y, _) ← normalized_plane[i]
        (u, v, _) ← normalized_image[i]
        Q[(2 * i)] ← [X, Y, 1.0, 0.0, 0.0, 0.0, ((−u) * X), ((−u) * Y), (−u)]
        Q[((2 * i) + 1)] ← [0.0, 0.0, 0.0, X, Y, 1.0, ((−v) * X), ((−v) * Y), (−v)]
    END FOR
    RETURN Q
END PROCEDURE
FOR view IN views DO
    Q ← build_homography_dlt_matrix(view["normalized_plane"], view["normalized_image"])
    (_, singular_values_Q, Vt_Q) ← ARRAY.linalg.svd(Q)
    h ← Vt_Q[(−1)]
    H_normalized ← RESHAPE(h, 3, 3)
    view["Q"] ← Q
    view["h"] ← h
    view["dlt_singular_values"] ← singular_values_Q
    view["H_normalized"] ← H_normalized
END FOR
CALL EMIT("Normalized DLT solved for all views.")
```

### Algorithm 7 — 6. Denormalize Each Homography

```text
PROCEDURE denormalize_homography_matrix(H_normalized, T_image, T_plane)
    H ← ((ARRAY.linalg.inv(T_image) · H_normalized) · T_plane)
    IF (ABS(H[(2, 2)]) < 1e-12) THEN
        RAISE ValueError("Degenerate homography: H[2, 2] is too close to zero.")
    END IF
    H ← (H / H[(2, 2)])
    RETURN H
END PROCEDURE
expected_corner_count ← (INTERNAL_CORNERS_X * INTERNAL_CORNERS_Y)
FOR view IN views DO
    view["H"] ← denormalize_homography_matrix(view["H_normalized"], view["T_image"], view["T_plane"])
    IF (view["im_pts"].shape ≠ (expected_corner_count, 2)) THEN
        RAISE ValueError(FORMAT("Unexpected image-point shape for {0}." , view["name"]))
    END IF
    IF (view["plane_pts"].shape ≠ (expected_corner_count, 2)) THEN
        RAISE ValueError(FORMAT("Unexpected planar-point shape for {0}." , view["name"]))
    END IF
    IF (NOT ARRAY.all(ARRAY.isfinite(view["H"]))) THEN
        RAISE ValueError(FORMAT("Non-finite homography for {0}." , view["name"]))
    END IF
END FOR
CALL EMIT("All homographies denormalized and validated.")
```

### Algorithm 8 — 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

```text
PROCEDURE build_zhang_constraint_vector(hi, hj)
    RETURN ARRAY.array([(hi[0] * hj[0]), ((hi[0] * hj[1]) + (hi[1] * hj[0])), (hi[1] * hj[1]),
        ((hi[2] * hj[0]) + (hi[0] * hj[2])), ((hi[2] * hj[1]) + (hi[1] * hj[2])), (hi[2] * hj[2])],
        dtype = REAL)
END PROCEDURE
PROCEDURE build_zhang_constraints_for_view(H)
    h1 ← H[(:, 0)]
    h2 ← H[(:, 1)]
    v12 ← build_zhang_constraint_vector(h1, h2)
    v11 ← build_zhang_constraint_vector(h1, h1)
    v22 ← build_zhang_constraint_vector(h2, h2)
    RETURN ARRAY.vstack((v12, (v11 − v22)))
END PROCEDURE
V ← ARRAY.vstack(LIST[build_zhang_constraints_for_view(view["H"]) FOR view IN views])
(_, singular_values, Vt) ← ARRAY.linalg.svd(V)
b ← Vt[(−1)]
CALL EMIT("Constraint matrix shape:", V.shape)
CALL EMIT("\nSingular values:")
CALL EMIT(singular_values)
CALL EMIT("\nResidual norm ||Vb||:")
CALL EMIT(ARRAY.linalg.norm((V · b)))
CALL EMIT("\nEstimated b vector:")
CALL EMIT(b)
```

### Algorithm 9 — 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

```text
(b11, b12, b22, b13, b23, b33) ← b
denominator ← ((b11 * b22) − (b12 ^ 2))
IF (ABS(denominator) < 1e-12) THEN
    RAISE ValueError("Degenerate intrinsic calibration constraints.")
END IF
v0 ← (((b12 * b13) − (b11 * b23)) / denominator)
lambda_ ← (b33 − (((b13 ^ 2) + (v0 * ((b12 * b13) − (b11 * b23)))) / b11))
IF (((lambda_ / b11) ≤ 0) OR (((lambda_ * b11) / denominator) ≤ 0)) THEN
    b ← (−b)
    (b11, b12, b22, b13, b23, b33) ← b
    denominator ← ((b11 * b22) − (b12 ^ 2))
    v0 ← (((b12 * b13) − (b11 * b23)) / denominator)
    lambda_ ← (b33 − (((b13 ^ 2) + (v0 * ((b12 * b13) − (b11 * b23)))) / b11))
END IF
alpha ← ARRAY.sqrt((lambda_ / b11))
beta ← ARRAY.sqrt(((lambda_ * b11) / denominator))
gamma ← ((((−b12) * (alpha ^ 2)) * beta) / lambda_)
u0 ← (((gamma * v0) / beta) − ((b13 * (alpha ^ 2)) / lambda_))
K ← ARRAY.array([[alpha, gamma, u0], [0.0, beta, v0], [0.0, 0.0, 1.0]], dtype = REAL)
CALL EMIT("Intrinsic parameters")
CALL EMIT("--------------------")
CALL EMIT(FORMAT("alpha : {0:.6f}" , alpha))
CALL EMIT(FORMAT("beta  : {0:.6f}" , beta))
CALL EMIT(FORMAT("gamma : {0:.6f}" , gamma))
CALL EMIT(FORMAT("u0    : {0:.6f}" , u0))
CALL EMIT(FORMAT("v0    : {0:.6f}" , v0))
CALL EMIT("\nIntrinsic matrix K:")
CALL EMIT(K)
```

### Algorithm 10 — 9. Recover $R$ and $t$ for Every Retained View

```text
PROCEDURE recover_camera_pose(K, H)
    K_inv ← ARRAY.linalg.inv(K)
    h1 ← H[(:, 0)]
    h2 ← H[(:, 1)]
    h3 ← H[(:, 2)]
    lambda_p ← (1.0 / ARRAY.linalg.norm((K_inv · h1)))
    r1 ← (lambda_p * (K_inv · h1))
    r2 ← (lambda_p * (K_inv · h2))
    r3 ← ARRAY.cross(r1, r2)
    R_approx ← ARRAY.column_stack((r1, r2, r3))
    (U, _, Vt) ← ARRAY.linalg.svd(R_approx)
    R ← (U · Vt)
    IF (ARRAY.linalg.det(R) < 0) THEN
        U[(:, (−1))] ← (U[(:, (−1))] * (−1))
        R ← (U · Vt)
    END IF
    t ← (lambda_p * (K_inv · h3))
    RETURN (R, t)
END PROCEDURE
FOR view IN views DO
    (R, t) ← recover_camera_pose(K, view["H"])
    view["R"] ← R
    view["t"] ← t
    CALL EMIT(view["name"])
    CALL EMIT("R =")
    CALL EMIT(R)
    CALL EMIT("t =", t)
    CALL EMIT(("-" * 60))
END FOR
```

### Algorithm 11 — 10. Reproject the $Z=0$ Calibration Points

```text
FOR view IN views DO
    chessboard_xy_points ← view["plane_pts"]
    chessboard_xyz_points ← ARRAY.column_stack((chessboard_xy_points,
        ARRAY.zeros(chessboard_xy_points.shape[0])))
    camera_frame_points ← ((chessboard_xyz_points · TRANSPOSE(view["R"])) + view["t"])
    projected_h ← (camera_frame_points · TRANSPOSE(K))
    IF ARRAY.any((ARRAY.abs(projected_h[(:, 2)]) < 1e-12)) THEN
        RAISE ValueError(FORMAT("Invalid homogeneous depth during reprojection for {0}." , view["name"]))
    END IF
    reprojected_image_points ← (projected_h[(:, :2)] / projected_h[(:, 2:3)])
    view["world_xyz"] ← chessboard_xyz_points
    view["projected"] ← reprojected_image_points
END FOR
CALL EMIT(FORMAT("Reprojected calibration views: {0}" , LENGTH(views)))
```

### Algorithm 12 — 11. Compute Point-wise Errors, Mean Error and RMSE

```text
FOR view IN views DO
    residuals ← (view["projected"] − view["im_pts"])
    errors ← ARRAY.linalg.norm(residuals, axis = 1)
    view["residuals"] ← residuals
    view["errors"] ← errors
    view["mean_error"] ← REAL(MEAN(errors))
    view["rmse"] ← REAL(ARRAY.sqrt(MEAN(ARRAY, (errors ^ 2))))
    CALL EMIT(view["name"])
    CALL EMIT(FORMAT("Mean reprojection error : {0:.4f} px" , view["mean_error"]))
    CALL EMIT(FORMAT("Reprojection RMSE       : {0:.4f} px" , view["rmse"]))
    CALL EMIT(("-" * 60))
END FOR
all_reprojection_errors ← ARRAY.concatenate(LIST[view["errors"] FOR view IN views])
overall_mean_error ← REAL(MEAN(all_reprojection_errors))
overall_rmse ← REAL(ARRAY.sqrt(MEAN(ARRAY, (all_reprojection_errors ^ 2))))
CALL EMIT("Calibration summary")
CALL EMIT("-------------------")
CALL EMIT(FORMAT("Valid calibration views : {0}" , LENGTH(views)))
CALL EMIT(FORMAT("Calibration points       : {0}" , LENGTH(all_reprojection_errors)))
CALL EMIT(FORMAT("Mean reprojection error  : {0:.4f} px" , overall_mean_error))
CALL EMIT(FORMAT("Overall RMSE             : {0:.4f} px" , overall_rmse))
```

### Algorithm 13 — 12. Produce and Save the Six Required Diagnostic Figures

```text
REQUIRED_OUTPUTS ← ["detected_chessboard_corners.png", "homography_estimation_pipeline.png", "estimated_camera_poses.png",
    "reprojection_results.png", "mean_reprojection_error_by_view.png",
    "reprojection_error_distribution.png"]
CALL EMIT("Diagnostic figures to generate:")
FOR output_name IN REQUIRED_OUTPUTS DO
    CALL EMIT(FORMAT("  - {0}" , output_name))
END FOR
```

### Algorithm 14 — 12.1 Homography Estimation Pipeline

```text
(fig, ax) ← PLOT.subplots(figsize = (15, 3.5))
CALL ax.axis("off")
labels ← ["World points\n(X, Y, 1)", "Image points\n(u, v, 1)", "Normalize\nT_plane, T_image", "Build Q\nQh = 0",
    "SVD\nSolve for h", "Denormalize\nH"]
x_positions ← ARRAY.linspace(0.08, 0.92, LENGTH(labels))
FOR (x, label) IN ZIP(x_positions, labels) DO
    CALL ax.text(x, 0.5, label, ha = "center", va = "center", fontsize = 12, transform = ax.transAxes,
        bbox = {"boxstyle" ↦ "round,pad=0.5", "fill" ↦ FALSE, "linewidth" ↦ 1.5})
END FOR
FOR (start, end) IN ZIP(x_positions[:(−1)], x_positions[1:]) DO
    CALL ax.annotate("", xy = ((end − 0.055), 0.5), xytext = ((start + 0.055), 0.5), xycoords = ax.transAxes,
        arrowprops = {"arrowstyle" ↦ "->", "linewidth" ↦ 1.5})
END FOR
CALL ax.set_title("Homography Estimation Pipeline", fontsize = 16, pad = 20)
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "homography_estimation_pipeline.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 15 — 12.2 Mean Reprojection Error by View

```text
image_names ← LIST[view["name"] FOR view IN views]
mean_reprojection_errors ← LIST[view["mean_error"] FOR view IN views]
(fig, ax) ← PLOT.subplots(figsize = (10, 5))
bars ← ax.bar(image_names, mean_reprojection_errors)
CALL ax.axhline(overall_mean_error, linestyle = "--", linewidth = 1.5,
    label = FORMAT("Overall mean = {0:.3f} px" , overall_mean_error))
FOR (bar, value) IN ZIP(bars, mean_reprojection_errors) DO
    CALL ax.text((bar.get_x() + (bar.get_width() / 2)), bar.get_height(), FORMAT("{0:.2f}" , value), ha = "center",
        va = "bottom", fontsize = 9)
END FOR
CALL ax.set_title("Mean Reprojection Error by Calibration View")
CALL ax.set_xlabel("Calibration image")
CALL ax.set_ylabel("Mean reprojection error (pixels)")
CALL ax.grid(axis = "y", alpha = 0.25)
CALL ax.legend()
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "mean_reprojection_error_by_view.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 16 — 12.3 Reprojection Error Distribution

```text
(fig, ax) ← PLOT.subplots(figsize = (9, 5))
CALL ax.hist(all_reprojection_errors, bins = 20, edgecolor = "black", alpha = 0.8)
CALL ax.axvline(overall_mean_error, linestyle = "--", linewidth = 1.5, label = FORMAT("Mean = {0:.3f} px" ,
    overall_mean_error))
CALL ax.axvline(overall_rmse, linestyle = ":", linewidth = 1.5, label = FORMAT("RMSE = {0:.3f} px" ,
    overall_rmse))
CALL ax.set_title("Distribution of Reprojection Errors")
CALL ax.set_xlabel("Reprojection error (pixels)")
CALL ax.set_ylabel("Number of calibration points")
CALL ax.grid(axis = "y", alpha = 0.25)
CALL ax.legend()
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "reprojection_error_distribution.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 17 — 12.4 Detected Chessboard Corners

```text
number_of_views ← LENGTH(views)
plot_columns ← 3
plot_rows ← INTEGER(ARRAY.ceil((number_of_views / plot_columns)))
(fig, axes) ← PLOT.subplots(plot_rows, plot_columns, figsize = (15, (5 * plot_rows)))
axes ← RESHAPE(ARRAY.asarray(axes), (−1))
FOR (ax, view) IN ZIP(axes, views) DO
    visual ← COPY(view["image"])
    corners ← CAST(RESHAPE(view["im_pts"], (−1), 1, 2), ARRAY.float32)
    CALL CV.drawChessboardCorners(visual, (INTERNAL_CORNERS_X, INTERNAL_CORNERS_Y), corners, TRUE)
    visual_rgb ← CV.cvtColor(visual, CV.COLOR_BGR2RGB)
    CALL ax.imshow(visual_rgb)
    CALL ax.set_title(view["name"])
    CALL ax.axis("off")
END FOR
FOR ax IN axes[number_of_views:] DO
    CALL ax.axis("off")
END FOR
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "detected_chessboard_corners.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 18 — 12.5 Estimated Camera Poses

```text
fig ← PLOT.figure(figsize = (10, 8))
ax ← fig.add_subplot(111, projection = "3d")
reference_chessboard_xy_points ← views[0]["plane_pts"]
reference_chessboard_xyz_points ← ARRAY.column_stack((reference_chessboard_xy_points,
    ARRAY.zeros(reference_chessboard_xy_points.shape[0])))
CALL ax.scatter(reference_chessboard_xyz_points[(:, 0)], reference_chessboard_xyz_points[(:, 1)],
    reference_chessboard_xyz_points[(:, 2)], marker = "s", label = "Chessboard")
FOR view IN views DO
    R ← view["R"]
    t ← view["t"]
    camera_center ← ((−TRANSPOSE(R)) · t)
    CALL ax.scatter(camera_center[0], camera_center[1], camera_center[2], marker = "o")
    CALL ax.text(camera_center[0], camera_center[1], camera_center[2], view["path"].stem, fontsize = 8)
END FOR
CALL ax.set_title("Estimated Camera Poses")
CALL ax.set_xlabel("X (m)")
CALL ax.set_ylabel("Y (m)")
CALL ax.set_zlabel("Z (m)")
CALL ax.legend()
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "estimated_camera_poses.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 19 — 12.6 Detected vs Reprojected Points

```text
number_of_views ← LENGTH(views)
plot_columns ← 3
plot_rows ← INTEGER(ARRAY.ceil((number_of_views / plot_columns)))
(fig, axes) ← PLOT.subplots(plot_rows, plot_columns, figsize = (15, (5 * plot_rows)))
axes ← RESHAPE(ARRAY.asarray(axes), (−1))
FOR (ax, view) IN ZIP(axes, views) DO
    image_rgb ← CV.cvtColor(view["image"], CV.COLOR_BGR2RGB)
    CALL ax.imshow(image_rgb)
    CALL ax.scatter(view["im_pts"][(:, 0)], view["im_pts"][(:, 1)], s = 24, marker = "o",
        label = "Detected")
    CALL ax.scatter(view["projected"][(:, 0)], view["projected"][(:, 1)], s = 24, marker = "x",
        label = "Reprojected")
    CALL ax.set_title(FORMAT("{0} — RMSE: {1:.3f} px" , view["name"], view["rmse"]))
    CALL ax.axis("off")
    CALL ax.legend(fontsize = 8)
END FOR
FOR ax IN axes[number_of_views:] DO
    CALL ax.axis("off")
END FOR
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "reprojection_results.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 20 — 13. Run the Numerical and Output-file Validation Checks

```text
IF ((K.shape ≠ (3, 3)) OR (NOT ARRAY.all(ARRAY.isfinite(K)))) THEN
    RAISE ValueError("Invalid intrinsic matrix K.")
END IF
FOR view IN views DO
    IF (NOT ARRAY.all(ARRAY.isfinite(view["H"]))) THEN
        RAISE ValueError(FORMAT("Non-finite homography for {0}." , view["name"]))
    END IF
    R ← view["R"]
    IF (NOT ARRAY.allclose((TRANSPOSE(R) · R), ARRAY.eye(3), atol = 1e-06)) THEN
        RAISE ValueError(FORMAT("Non-orthonormal rotation for {0}." , view["name"]))
    END IF
    IF (NOT ARRAY.isclose(ARRAY.linalg.det(R), 1.0, atol = 1e-06)) THEN
        RAISE ValueError(FORMAT("Invalid rotation determinant for {0}." , view["name"]))
    END IF
    IF (NOT ARRAY.all(ARRAY.isfinite(view["errors"]))) THEN
        RAISE ValueError(FORMAT("Non-finite residuals for {0}." , view["name"]))
    END IF
END FOR
missing ← LIST[name FOR name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / name).exists())]
IF missing THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing)))
END IF
CALL EMIT("All Camera Calibration validation checks passed.")
```

### Completeness check

| Executable element | Included in the pseudocode |
| --- | ---: |
| Source code cells | 23 |
| Procedure bodies | 9 |
| Assignments | 146 |
| Conditional branches | 22 |
| Loops | 22 |
| Assertions | 0 |
| Explicit error branches | 17 |

The counts check transcription coverage, not mathematical correctness or a successful execution. Every original diagnostic and rendering call is included. The limitations discussed below still apply.

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Load the Sorted JPEG Calibration Images
- **Task 2:** Detect and Refine Chessboard Corners
- **Task 3:** Build the Planar World Coordinates
- **Task 4:** Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$
- **Task 5:** Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD
- **Task 6:** Denormalize Each Homography
- **Task 7:** Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD
- **Task 8:** Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$
- **Task 9:** Recover $R$ and $t$ for Every Retained View
- **Task 10:** Reproject the $Z=0$ Calibration Points
- **Task 11:** Compute Point-wise Errors, Mean Error and RMSE
- **Task 12:** Produce and Save the Six Required Diagnostic Figures
- **Task 13:** Run the Numerical and Output-file Validation Checks


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation** of this algorithm.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 6 files under `../outputs/figures/`:

- `detected_chessboard_corners.png`
- `homography_estimation_pipeline.png`
- `estimated_camera_poses.png`
- `reprojection_results.png`
- `mean_reprojection_error_by_view.png`
- `reprojection_error_distribution.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/computer-vision/camera-calibration/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `detect_and_refine_chessboard_corners`
- `build_chessboard_plane_points`
- `convert_to_homogeneous_coordinates`
- `compute_point_normalization_transform`
- `build_homography_dlt_matrix`
- `denormalize_homography_matrix`
- `build_zhang_constraint_vector`
- `build_zhang_constraints_for_view`
- `recover_camera_pose`

## Implementation Limits and Interpretation

- The implementation uses normalized DLT points but **does not row-normalize Zhang's V** or apply spectral-gap rejection. Row normalization changes equation weighting; it is not proof of physical correctness and does not force noisy residuals to machine precision.
- A near-zero final singular value is expected for exact homogeneous systems. If conditioning diagnostics are added later, distinguish the nullspace gap from conditioning in the constrained subspace; do not reject an exact fit using `sigma_max/sigma_min > 1e6`.
- The intrinsic denominator threshold is absolute and scale-dependent. Global sign reversal leaves the closed-form radicands unchanged. No Cholesky extraction or automatic repair of an indefinite conic is present.
- Rotation projection is implemented, including determinant correction. The in-place update of the local SVD factor is valid; it is not a demonstrated NumPy bug.
- Positive focal lengths, positive depth for **every** board point and finite translations require review. The current code does not enforce cheirality, and `t_z>0` alone would not prove all board points are in front.
- The model excludes lens distortion and nonlinear refinement. No universal RMSE acceptance threshold, JSON audit export or new package/module is introduced by this algorithm update.

## 17. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.